# AfriVoices — AUDIT D'ERREURS (v8-cible) sur le dev officiel

**But** : arrêter de spéculer, REGARDER les erreurs réelles. On transcrit le dev officiel
avec v8-cible + KenLM v2, on aligne prédiction vs référence, et on catégorise le WER :
- **normalisation** (chiffres, ponctuation, casse, apostrophes) → corrigeable SANS réentraîner
- **substitutions proches** (1-2 caractères : orthographe/diacritiques) → post-traitement possible
- **erreurs franches** (mot totalement faux) → vrai problème acoustique
- **insertions / suppressions** → segmentation

Sortie : un tableau par langue (part de chaque type d'erreur) + des exemples concrets.
Si une grosse part est "normalisation/proche", un simple nettoyage gagne du WER d'un coup.

**Session GPU fraîche.** §1 installe kenlm+pyctcdecode → REDÉMARRER → §2...

## 1. Dépendances (redémarrer le runtime après)

In [ ]:
import importlib, subprocess
need=[m for m in ["kenlm","pyctcdecode","jiwer"] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run(["pip","install","-q","pyctcdecode","jiwer"], check=False)
    subprocess.run(["pip","install","-q","https://github.com/kpu/kenlm/archive/master.zip"], check=False)
    print("⚠️ REDÉMARRE le runtime puis relance à partir de §2")
else:
    print("✓ prêt")

## 2. Drive + reconstruction du dev officiel (depuis anvke/)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, glob, re, io, base64, numpy as np, soundfile as sf, librosa
import torch
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor

BASE="/content/drive/MyDrive/afrivoices"
MODEL=f"{BASE}/baobab-asr-v8-cible"
LM=f"{BASE}/lm_v2"
ALPHA,BETA=0.7,0.5
dev="cuda" if torch.cuda.is_available() else "cpu"

model=Wav2Vec2BertForCTC.from_pretrained(MODEL, local_files_only=True).to(dev).eval()
processor=Wav2Vec2BertProcessor.from_pretrained(MODEL, local_files_only=True)
tokenizer=processor.tokenizer

def clean_text(t):
    t=(t or "").lower().strip()
    t=re.sub(r"[^\w\s\u0129\u0169\u00e1\u00e0\u00e2\u00e4\u00e9\u00e8\u00ea\u00eb\u00ed\u00ec\u00ee\u00ef\u00f3\u00f2\u00f4\u00f6\u00fa\u00f9\u00fb\u00fc\']","",t)
    return re.sub(r"\s+"," ",t)

ANVKE=f"{BASE}/anvke"
ALIAS={"kik":["kik","kikuyu","gikuyu"],"luo":["luo","dholuo"],"kln":["kln","kalenjin"],
       "mas":["mas","maasai"],"som":["som","somali"]}
lang_dirs={}
for d in os.listdir(ANVKE):
    dl=d.lower()
    for lang,als in ALIAS.items():
        if any(a in dl for a in als) and lang not in lang_dirs:
            lang_dirs[lang]=os.path.join(ANVKE,d)
print("langues:", list(lang_dirs))

In [ ]:
from datasets import load_dataset, Audio

def duree_bytes(b):
    try: i=sf.info(io.BytesIO(b)); return i.frames/i.samplerate
    except Exception:
        try: i=sf.info(io.BytesIO(base64.b64decode(b))); return i.frames/i.samplerate
        except Exception: return 999.0

# dev officiel : 1er scripted + 1er unscripted, ~60/langue (identique aux évals de training)
eval_items={}
for lang,root in lang_dirs.items():
    allp=sorted(glob.glob(f"{root}/**/*.parquet", recursive=True))
    dv=[p for p in allp if "/dev/" in p or os.path.basename(p).startswith("dev")]
    picks=[p for p in dv if "unscripted" not in os.path.basename(p)][:1] + \
          [p for p in dv if "unscripted" in os.path.basename(p)][:1]
    if not picks: picks=dv[:2] if dv else allp[:1]
    d=load_dataset("parquet", data_files=picks, split="train").cast_column("audio", Audio(decode=False))
    rows=[]
    for ex in d.shuffle(seed=42):
        if len(rows)>=60: break
        t=(ex.get("transcription") or "").strip()
        b=ex["audio"].get("bytes") if isinstance(ex["audio"],dict) else None
        if not t or not b: continue
        if duree_bytes(b)>20: continue
        rows.append((b, clean_text(t)))
    eval_items[lang]=rows
    print(f"{lang}: {len(rows)} items dev")

## 3. Décodeurs KenLM + transcription du dev

In [ ]:
from pyctcdecode import build_ctcdecoder
from collections import Counter

raw=[t for t,_ in sorted(tokenizer.get_vocab().items(), key=lambda kv: kv[1])]
blank=tokenizer.pad_token
labels=[]; n=0
for t in raw:
    if t==blank: labels.append("")
    elif t in ("|"," "): labels.append(" ")
    elif t in {"[UNK]","<s>","</s>"}: n+=1; labels.append("\u2047"*n)
    else: labels.append(t)

def unigrams(lang, top=50000):
    c=Counter()
    for line in open(f"{LM}/{lang}.txt", encoding="utf-8"): c.update(line.split())
    return [w for w,_ in c.most_common(top)]

def decode_bytes(b):
    try: arr,sr=sf.read(io.BytesIO(b),dtype="float32")
    except Exception: arr,sr=sf.read(io.BytesIO(base64.b64decode(b)),dtype="float32")
    if arr.ndim>1: arr=arr.mean(axis=1)
    if sr!=16000: arr=librosa.resample(arr,orig_sr=sr,target_sr=16000)
    return arr.astype(np.float32)

decoders={l: build_ctcdecoder(labels, kenlm_model_path=f"{LM}/{l}.bin",
                              unigrams=unigrams(l), alpha=ALPHA, beta=BETA) for l in eval_items}

pairs={}   # lang -> [(ref, pred), ...]
with torch.no_grad():
    for lang, rows in eval_items.items():
        out=[]
        for b, ref in rows:
            arr=decode_bytes(b)
            fe=processor.feature_extractor([arr], sampling_rate=16000, return_tensors="pt", padding=True)
            lg=model(**{k:v.to(dev) for k,v in fe.items()}).logits[0].cpu().numpy()
            out.append((ref, decoders[lang].decode(lg)))
        pairs[lang]=out
        print(f"{lang}: {len(out)} transcrits")

## 4. Catégorisation des erreurs (le cœur de l'audit)

Pour chaque paire (ref, pred), on aligne mot à mot et on classe chaque écart :
- **num** : un des deux côtés est un nombre en chiffres, l'autre en lettres (ou ponctuation)
- **proche** : les 2 mots diffèrent de ≤ 2 caractères (orthographe/diacritique)
- **franche** : substitution où les mots n'ont presque rien en commun (vraie erreur d'écoute)
- **insertion / suppression** : mot en trop / manquant

In [ ]:
import jiwer
from difflib import SequenceMatcher

def lev(a,b):
    # distance de Levenshtein simple
    if a==b: return 0
    m,n=len(a),len(b)
    dp=list(range(n+1))
    for i in range(1,m+1):
        prev=dp[0]; dp[0]=i
        for j in range(1,n+1):
            cur=dp[j]
            dp[j]=min(dp[j]+1, dp[j-1]+1, prev+(a[i-1]!=b[j-1]))
            prev=cur
    return dp[n]

def categorise(ref, pred):
    r, p = ref.split(), pred.split()
    sm=SequenceMatcher(None, r, p)
    cats=Counter()
    for tag,i1,i2,j1,j2 in sm.get_opcodes():
        if tag=="equal": continue
        if tag=="insert": cats["insertion"]+=(j2-j1); continue
        if tag=="delete": cats["suppression"]+=(i2-i1); continue
        # replace : apparier mot à mot
        for k in range(max(i2-i1, j2-j1)):
            rw=r[i1+k] if i1+k<i2 else ""
            pw=p[j1+k] if j1+k<j2 else ""
            if not rw: cats["insertion"]+=1; continue
            if not pw: cats["suppression"]+=1; continue
            if (rw.isdigit()!=pw.isdigit()):
                cats["num"]+=1
            elif lev(rw,pw)<=2:
                cats["proche"]+=1
            else:
                cats["franche"]+=1
    return cats

print(f"{'langue':6} {'WER':>6} | {'num':>5} {'proche':>7} {'franche':>8} {'ins':>5} {'sup':>5}  (part du total d'erreurs)")
global_cat=Counter()
for lang, out in pairs.items():
    refs=[r for r,_ in out]; preds=[p for _,p in out]
    wer=jiwer.wer(refs, preds)
    cats=Counter()
    for r,p in out:
        for k,v in categorise(r,p).items(): cats[k]+=v; global_cat[k]+=v
    tot=sum(cats.values()) or 1
    print(f"{lang:6} {wer:6.3f} | "
          f"{cats['num']/tot:5.0%} {cats['proche']/tot:7.0%} {cats['franche']/tot:8.0%} "
          f"{cats['insertion']/tot:5.0%} {cats['suppression']/tot:5.0%}")
tot=sum(global_cat.values()) or 1
print("\n=== GLOBAL ===")
for k in ["num","proche","franche","insertion","suppression"]:
    print(f"  {k:12}: {global_cat[k]/tot:5.1%}")
print(f"\nLecture: 'num'+'proche' élevés = gains possibles par NORMALISATION (sans réentraîner).")
print(f"         'franche' élevé = vrai problème acoustique (seul un meilleur modèle aide).")

## 5. Exemples concrets (pour voir de tes yeux)

In [ ]:
import jiwer
for lang, out in pairs.items():
    print(f"\n===== {lang} =====")
    # trie par WER décroissant pour voir les pires
    scored=sorted(out, key=lambda rp: jiwer.wer([rp[0]],[rp[1]]), reverse=True)
    for ref, pred in scored[:5]:
        print(f"  REF : {ref[:100]}")
        print(f"  PRED: {pred[:100]}")
        print()

## 6. Que faire des résultats

- Si **num + proche > ~30%** du total : un post-traitement (normaliser chiffres↔lettres,
  gérer les diacritiques/variantes) peut récupérer une bonne part → gain leaderboard SANS
  réentraîner. On écrit la fonction de normalisation ensemble et tu resoumets.
- Si **franche domine (>50%)** : le modèle entend mal → seul un meilleur backbone
  (omniASR fine-tuné) fera bouger les choses. Décision architecture.
- Regarde surtout le **somali** (variantes orthographiques connues) et le **kln/mas**
  (langues les plus dures) : c'est là que se cache le gros du WER (61% du test).